# Production Concerns

**Module:** 14 — AI Orchestration

SLOs, observability, cost, and safety for orchestrated AI.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Define SLOs for orchestrated AI runs
- Instrument traces/metrics/logs with run correlation
- Apply cost guards and safety checks in the control plane


## Reliability & SLOs

### Definition
Service level objectives for AI workflows: success rate, latency, human-queue wait, cost per successful run.

### Why it matters
Without SLOs, failures are anecdotes. With them, you can prioritize engineering.

### How it works
Pick a few user-centric metrics; error budgets drive freeze/ship decisions; separate model quality SLOs from infra SLOs.

### Intuition
If you can't measure 'good,' you can't operate it.

### Pitfalls
- Vanity metrics only
- One giant SLO that mixes unlike workloads
- Ignoring waiting_human time

### When to use
Production traffic of any volume.


| SLO | Example target | Notes |
|-----|----------------|-------|
| Run success rate | 99% / 30d | Exclude user cancels |
| p95 time-to-first-draft | < 8s | Interactive |
| p95 full completion (auto) | < 45s | No HITL |
| HITL queue wait p95 | < 15m | Business hours |
| Cost / successful run | < $0.12 | By workflow version |
| Safety incident rate | 0 critical / quarter | Override class |

```mermaid
flowchart LR
  Run --> Trace
  Run --> Metrics
  Run --> Logs
  Trace --> Alerting
  Metrics --> Alerting
```


In [ ]:
# Demo 1: SLO evaluation on a sample
runs = [
    {"ok": True, "ms": 1200},
    {"ok": True, "ms": 4000},
    {"ok": False, "ms": 9000},
    {"ok": True, "ms": 7000},
    {"ok": True, "ms": 1500},
]

def pct(vals, p):
    s = sorted(vals)
    idx = min(len(s) - 1, int(round((p / 100) * (len(s) - 1))))
    return s[idx]

success = sum(r["ok"] for r in runs) / len(runs)
p95 = pct([r["ms"] for r in runs], 95)
print({"success": success, "p95_ms": p95, "slo_success_ok": success >= 0.99, "slo_p95_ok": p95 <= 8000})


## Observability

### Definition
Traces, metrics, and logs correlated by `run_id` / `trace_id` across model/tool/human steps.

### Why it matters
Orchestrated systems fail in the seams — you need seam-visible telemetry.

### How it works
OpenTelemetry-style spans per node; attributes for model, tokens, tool name; structured logs; redaction.

### Intuition
Distributed tracing for prompts and tools.

### Pitfalls
- Logging full prompts with secrets
- No link from user ticket → run_id
- Metrics without labels for workflow_version

### When to use
From the first staging deploy.


In [ ]:
# Demo 2: structured span logger
import time, json

class Span:
    def __init__(self, name, run_id, **attrs):
        self.name, self.run_id, self.attrs = name, run_id, attrs
        self.t0 = None
    def __enter__(self):
        self.t0 = time.time()
        return self
    def __exit__(self, *exc):
        rec = {
            "span": self.name,
            "run_id": self.run_id,
            "ms": int((time.time() - self.t0) * 1000),
            "error": str(exc[1]) if exc[1] else None,
            **self.attrs,
        }
        print(json.dumps(rec))

with Span("retrieve", "r1", tool="search_kb"):
    time.sleep(0.01)
with Span("draft", "r1", model="gpt-4o-mini", prompt_tokens=300):
    time.sleep(0.01)


In [ ]:
# Demo 3: cost guard mid-run
class Budget:
    def __init__(self, max_usd: float):
        self.max_usd = max_usd
        self.spent = 0.0
    def charge(self, usd: float):
        if self.spent + usd > self.max_usd:
            raise RuntimeError("budget exceeded")
        self.spent += usd

b = Budget(0.05)
for cost in [0.02, 0.02, 0.02]:
    try:
        b.charge(cost)
        print("charged", cost, "total", round(b.spent, 4))
    except RuntimeError as e:
        print("blocked", e)


In [ ]:
# Demo 4: safety check before side effect
FORBIDDEN = {"delete_prod", "wire_transfer"}

def before_tool(tool: str, args: dict, policy_allow: set[str]):
    if tool in FORBIDDEN:
        return "deny: forbidden tool"
    if tool not in policy_allow:
        return "deny: not allow-listed"
    if tool == "issue_refund" and args.get("amount", 0) > 500:
        return "deny: amount requires human"
    return "allow"

print(before_tool("issue_refund", {"amount": 900}, {"issue_refund", "search_kb"}))
print(before_tool("search_kb", {}, {"issue_refund", "search_kb"}))


In [ ]:
# Demo 5: alert rule sketch
def should_page(success_rate_1h: float, p95_ms: float, budget_burn: float) -> list[str]:
    alerts = []
    if success_rate_1h < 0.95:
        alerts.append("page: success SLO burn")
    if p95_ms > 15000:
        alerts.append("ticket: latency elevated")
    if budget_burn > 1.5:
        alerts.append("page: cost anomaly")
    return alerts

print(should_page(0.92, 9000, 1.1))


### Try it yourself — Production

1. Define 4 SLOs for a research agent product.
2. Add `workflow_version` label to the span logger.
3. Write a redaction function for span attributes named `prompt`.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `SLO` | Service level objective |
| `error budget` | Allowed unreliability |
| `span` | Timed unit of work in a trace |
| `budget burn` | Spend vs expected rate |


## Incident Playbooks (short)

| Symptom | First checks |
|---------|--------------|
| Success SLO burn | Error breakdown by node/tool; dependency status |
| Latency spike | Model provider status; retrieve p95; queue depth |
| Cost anomaly | Top runs by tokens; runaway agent loops; top_k blowups |
| Safety near-miss | Tool allow-list logs; HITL bypass bugs |

### Error budget policy
If monthly error budget exhausted → feature freeze on risky workflows; reliability work only.


In [ ]:
# Token anomaly detector
import statistics

daily_tokens = [1e5, 1.1e5, 9e4, 1.2e5, 5e5]
mu = statistics.mean(daily_tokens[:-1])
sigma = statistics.pstdev(daily_tokens[:-1]) or 1.0
z = (daily_tokens[-1] - mu) / sigma
print({"mu": mu, "sigma": sigma, "z": round(z, 2), "alert": z > 3})


In [ ]:
# Redact attributes before export
SENSITIVE_KEYS = {"prompt", "api_key", "ssn"}

def redact(attrs: dict) -> dict:
    out = {}
    for k, v in attrs.items():
        out[k] = "[REDACTED]" if k in SENSITIVE_KEYS else v
    return out

print(redact({"prompt": "SECRET", "model": "gpt", "run_id": "r1"}))


### Try it yourself — Production deepen

1. Define paging vs ticket thresholds for your Project 1 orchestrator.
2. Add workflow_version to the anomaly detector labels (group tokens by version).


## Key Takeaways

- Operate orchestration with SLOs
- Correlate everything by run_id
- Enforce cost and safety in control plane
- Alert on burn, not on single blips only
